# AI-POWERED CHATBOT

In [98]:
# install dependencies
!pip install sentence-transformers ipywidgets

In [57]:
# ============================================================
#  AI-Powered FAQ Chatbot 
# ============================================================

# Import libraries
import sqlite3
import uuid
import re
from datetime import datetime, timezone
from IPython.display import clear_output
from sentence_transformers import SentenceTransformer, util

print("⏳ Loading NLP model...")
model = SentenceTransformer('all-MiniLM-L6-v2')
print("✅ Model ready!\n")


⏳ Loading NLP model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Model ready!



In [66]:
# ─────────────────────────────────────────────
# Knowledge Base
# ─────────────────────────────────────────────

FAQ_DATA = [
    {"topic": "Refund Policy", "question": "What is your refund policy?", "answer": "We offer a full 30-day money-back guarantee. If you're not completely satisfied, just contact support and we'll process your refund — no questions asked!"},
    {"topic": "Password Reset", "question": "How do I reset my password?", "answer": "Easy! Click 'Forgot Password' on the login screen. We'll instantly email you a secure reset link."},
    {"topic": "Support Hours", "question": "What are your customer support hours?", "answer": "Our AI assistant is available 24/7. Human specialists are available via support@example.com from 9 AM – 5 PM EST."},
    {"topic": "International Shipping", "question": "Do you ship internationally?", "answer": "Yes! We ship to 50+ countries worldwide. Shipping fees and delivery times vary by destination."},
    {"topic": "Pricing & Plans", "question": "What are your pricing plans?", "answer": "We have three plans:\n• Basic — $9/month\n• Pro — $29/month\n• Enterprise — custom pricing\nAll plans come with a 30-day free trial!"},
    {"topic": "Cancel Subscription", "question": "How do I cancel my subscription?", "answer": "You can cancel anytime from your Account Dashboard → Billing → Cancel Plan. No hidden fees."},
    {"topic": "Exit Chat", "question": "How do I exit?", "answer": "To end the chat, simply type 'quit', 'exit', 'end', or 'stop'."}
]

# Pre-compute arrays for the menu and semantic matching
FAQ_QUESTIONS  = [item["question"] for item in FAQ_DATA]
FAQ_TOPICS     = [item["topic"] for item in FAQ_DATA]
FAQ_EMBEDDINGS = model.encode(FAQ_QUESTIONS, convert_to_tensor=True)

print("Knowledge base is loaded successfully.")

Knowledge base is loaded successfully.


In [91]:
# ─────────────────────────────────────────────
# SQLite Logging Setup
# ─────────────────────────────────────────────
DB_FILE = "faq_chatbot_logs.db"

def init_db():
    conn = sqlite3.connect(DB_FILE)
    conn.execute("""
        CREATE TABLE IF NOT EXISTS interaction_logs (
            id INTEGER PRIMARY KEY AUTOINCREMENT, 
            session_id TEXT, 
            user_message TEXT, 
            bot_response TEXT, 
            confidence_score REAL, 
            timestamp TEXT
        )
    """)
    conn.commit()
    conn.close()

def log_interaction(session_id, user_msg, bot_resp, score):
    conn = sqlite3.connect(DB_FILE)
    conn.execute("""
        INSERT INTO interaction_logs (session_id, user_message, bot_response, confidence_score, timestamp)
        VALUES (?, ?, ?, ?, ?)
    """, (session_id, user_msg, bot_resp, score, datetime.now(timezone.utc).isoformat()))
    conn.commit()
    conn.close()

init_db()
print("SQLite database is ready.")

SQLite database is ready.


In [100]:
# ─────────────────────────────────────────────
# Core Logic (Building CHATBOT)
# ─────────────────────────────────────────────
CONFIDENCE_THRESHOLD = 0.40

# set of greetings word
GREETINGS = {"hi", "hello", "hey", "hii", "hiii", "greetings", "good morning", "good afternoon", "good evening"}

# This array secretly stores whole conversation
display_history = []

def cprint(text):
    """Custom print function that displays text to the terminal AND saves it to our history array."""
    print(text)
    display_history.append(text)

def show_menu():
    """Prints the numbered 'Dropdown' menu to the terminal based on available FAQ topics."""
    cprint("\n📋 Here are some topics I can help you with:")
    for i, topic in enumerate(FAQ_TOPICS, 1):
        cprint(f"   {i}. {topic}")
    cprint("   (Type a number to select a topic, or just type your question normally!)")

def get_response(user_message: str):
    """
    Takes the user's message, converts it into an AI embedding, and compares it 
    to all known FAQ questions using cosine similarity to find the best match.
    """
    # Convert the text into a mathematical tensor (embedding)
    query_emb  = model.encode(user_message, convert_to_tensor=True)
    # Calculate similarity scores against the pre-computed FAQ database
    cos_scores = util.cos_sim(query_emb, FAQ_EMBEDDINGS)[0].tolist()
    
    # Find the index of the highest scoring match
    best_idx   = max(range(len(cos_scores)), key=lambda i: cos_scores[i])
    best_score = cos_scores[best_idx]

    # Return the answer if the score meets our minimum confidence threshold
    if best_score >= CONFIDENCE_THRESHOLD:
        return FAQ_DATA[best_idx]["answer"], best_score
    else:
        # Fallback message if the AI isn't confident enough
        return "Hmm, I'm not sure I have the right answer for that 🤔. Could you try rephrasing or picking a topic from the menu?", best_score

# ─────────────────────────────────────────────
# Main Chatbot Loop
# ─────────────────────────────────────────────

session_id = str(uuid.uuid4())[:8].upper()

cprint("=" * 60)
cprint(f"🤖 AI Assistant Started (Session: {session_id})")
cprint("=" * 60)

cprint("\n🤖 Bot: 👋 Hi there! I'm your AI FAQ Assistant.")
show_menu()

while True:
    try:
        # 1. Wait for user input and remove any extra spaces
        user_input = input("\n👤 You: ").strip()
        
        # 2. Ignore the input if the user just pressed Enter without typing anything
        if not user_input:
            continue
            
        # 3. Save the user's message to our custom history tracker
        display_history.append(f"\n👤 You: {user_input}")
            
        # 4. Check for hardcoded exit commands
        if user_input.lower() in ['quit', 'exit', 'stop', 'end']:
            cprint("\n🤖 Bot: Goodbye! Have a great day! 👋")
            cprint("── Chat session closed ──")
            break # Exit the while loop entirely
            
        # 5. Clean the input (remove punctuation) to check if it's just a greeting
        clean_input = re.sub(r'[^\w\s]', '', user_input.lower()).strip()
        if clean_input in GREETINGS:
            cprint("\n🤖 Bot: Hello again! 😊 How can I assist you today?")
            show_menu() # Re-display the menu to guide the user
            continue

        # 6. Handle explicit menu number selections (e.g., user types "1" or "4")
        if user_input.isdigit():
            selected_num = int(user_input)
            
            # Check if the number is within the valid range of topics
            if 1 <= selected_num <= len(FAQ_TOPICS):
                idx = selected_num - 1
                
                # if exit chat selected
                if FAQ_DATA[idx]["topic"] == "Exit Chat":
                    cprint("\n🤖 Bot: Goodbye! Have a great day! 👋")
                    cprint("── Chat session closed ──")
                    break 
                
                # Normal Case: Retrieve and print the matching FAQ answer
                answer = FAQ_DATA[idx]["answer"]
                cprint(f"\n🤖 Bot: {answer}")
                # Log the interaction with a perfect confidence score (1.0) since they explicitly chose it
                log_interaction(session_id, f"[Menu Selection: {FAQ_TOPICS[idx]}]", answer, 1.0)
                continue
                
            else:
                # Handle invalid numbers 
                cprint(f"\n🤖 Bot: Oops! '{selected_num}' isn't on the menu. Please pick a number between 1 and {len(FAQ_TOPICS)}.")
                continue
            
        # 7. If it's not a number or command, use the AI model to find an answer
        answer, score = get_response(user_input)
        
        # 8. Log the interaction to the SQLite database
        log_interaction(session_id, user_input, answer, score)
        
        # 9. Print the bots's response and confidence score
        cprint(f"\n🤖 Bot: {answer}")
        cprint(f"   [AI Confidence Score: {score:.2f}]")
        
    except KeyboardInterrupt:
        # 10. Handle interruptions
        clear_output(wait=False)
        for line in display_history:
            print(line)
        print("\n\n🤖 Bot: Session interrupted. Goodbye! 👋")
        break 

🤖 AI Assistant Started (Session: 783EA7FE)

🤖 Bot: 👋 Hi there! I'm your AI FAQ Assistant.

📋 Here are some topics I can help you with:
   1. Refund Policy
   2. Password Reset
   3. Support Hours
   4. International Shipping
   5. Pricing & Plans
   6. Cancel Subscription
   7. Exit Chat
   (Type a number to select a topic, or just type your question normally!)



👤 You:  Hii



🤖 Bot: Hello again! 😊 How can I assist you today?

📋 Here are some topics I can help you with:
   1. Refund Policy
   2. Password Reset
   3. Support Hours
   4. International Shipping
   5. Pricing & Plans
   6. Cancel Subscription
   7. Exit Chat
   (Type a number to select a topic, or just type your question normally!)



👤 You:  2



🤖 Bot: Easy! Click 'Forgot Password' on the login screen. We'll instantly email you a secure reset link.


...


🤖 Bot: We have three plans:
• Basic — $9/month
• Pro — $29/month
• Enterprise — custom pricing
All plans come with a 30-day free trial!



👤 You:  6



🤖 Bot: You can cancel anytime from your Account Dashboard → Billing → Cancel Plan. No hidden fees.



👤 You:  7



🤖 Bot: Goodbye! Have a great day! 👋
── Chat session closed ──


In [109]:
# ─────────────────────────────────────────────
# See records on SQLite Database
# ─────────────────────────────────────────────

# Connect to your database file
conn = sqlite3.connect("faq_chatbot_logs.db")

query = "SELECT * FROM interaction_logs ORDER BY id ASC"

df = pd.read_sql_query(query, conn)

conn.close()
display(df)

,id,session_id,selected_topic,user_message,bot_response,confidence_score,matched_faq,timestamp


In [107]:
# ─────────────────────────────────────────────
# Saved chat history to CSV File
# ─────────────────────────────────────────────

conn = sqlite3.connect("faq_chatbot_logs.db")
df = pd.read_sql_query("SELECT * FROM interaction_logs", conn)
conn.close()
df.to_csv("Chat_History_Export.csv", index=False)

print("Saved as CSV file!")

Saved as CSV file!


In [108]:
# ─────────────────────────────────────────────
# Delete all records from SQLite Database
# ─────────────────────────────────────────────

DB_FILE = "faq_chatbot_logs.db"

conn = sqlite3.connect(DB_FILE)

cursor = conn.cursor()
cursor.execute("DELETE FROM interaction_logs")
cursor.execute("DELETE FROM sqlite_sequence WHERE name='interaction_logs'")
conn.commit()

conn.close()
print("All chat logs have been successfully erased!")

All chat logs have been successfully erased!
